Below we initialize a list of dictionaries, each dictionary being a movie. This list will be used throughout the program to store and retrieve movie entries.


In [449]:
from datetime import datetime

In [450]:
MAX_BOOKINGS = 5
last_movie_id = 1005

In [451]:
movies = [
    {
        "id": 1000,
        "title": "Inception",
        "price": 10.0,
        "date": datetime(2026, 5, 1),
        "bookings": 0,
    },
    {
        "id": 1001,
        "title": "The Dark Knight",
        "price": 12.5,
        "date": datetime(2026, 5, 2),
        "bookings": 0,
    },
    {
        "id": 1002,
        "title": "Interstellar",
        "price": 11.0,
        "date": datetime(2026, 5, 3),
        "bookings": 5,
    },
    {
        "id": 1003,
        "title": "Avengers: Endgame",
        "price": 13.0,
        "date": datetime(2026, 5, 4),
        "bookings": 5,
    },
    {
        "id": 1004,
        "title": "Spider-Man: No Way Home",
        "price": 12.0,
        "date": datetime(2026, 5, 5),
        "bookings": 0,
    },
]

In [452]:
def prompt_confirm(message):
    input_confirm = input(f"\n{message} ([Y]es / [N]o) => ")
    if input_confirm.strip().lower() == "y":
        return True
    else:
        return False

In [453]:
def prompt_choice(message):
    while True:
        choice = input(f"\n{message}")
        try:
            choice = int(choice.strip())
            return choice
        except ValueError:
            print("Choose a valid option")

In [454]:
def prompt_date():
    while True:
        input_date = input("Enter movie date (DD-MM-YYYY) => ")
        try:
            input_date = datetime.strptime(input_date.strip(), "%d-%m-%Y")
            return input_date
        except ValueError:
            print("Input a valid date in the format DD-MM-YYYY")

In [455]:
def prompt_title():
    while True:
        input_title = input("Enter the Movie title => ").strip()
        if not input_title:
            print("Title can't be blank or only whitespaces")
        else:
            return input_title

In [456]:
def prompt_float(message):
    while True:
        input_float = input(message)
        try:
            input_float = float(input_float.strip())
            return input_float
        except ValueError:
            print("Enter a valid value")

In [457]:
def run_menu(menu_list, menu_title):
    while True:
        
        print("="*35)
        print(f" {menu_title}")
        print("="*35)
        for i, item in enumerate(menu_list, start=1):
            print(f" {i}. {item['entry_name']}")
        print(" 0. Back")
        choice = prompt_choice(" Choose an option => ")

        if choice == 0:
            break

        if choice < 0 or choice > len(menu_list):
            print("Enter a valid choice")
            continue
        print("_"*35)
        menu_list[choice - 1]["action"]()
        

In [458]:
def add_movie():
    new_movie = {
        "title": prompt_title(),
        "price": prompt_float("Enter price => "),
        "date": prompt_date(),
        "bookings": 0,
    }
    global last_movie_id
    last_movie_id += 1
    new_movie["id"] = last_movie_id
    movies.append(new_movie)

In [ ]:
def list_movies(show_booked=True):
    print(
        f"  {'Sn':>4.3} | {'ID':<6} | {'Title':<26.25} | {'Price':>7.6} | {'Date':<13} | {'Bookings':<9.8}"
    )
    print("=" * 85)

    """ This code checks if the argument show_booked is true. If it is true then the movies_list will reference the global
    movies variable. 
    If it is false, then a new list is generated filtering out all movies where bookings value exceeds or
    is equal to the global MAX_BOOKINGS variable. """
    movies_list = (
        movies
        if show_booked
        else [x for x in movies if x["bookings"] < MAX_BOOKINGS]
    )

    for index, item in enumerate(movies_list, start=1):
        formatted_date = datetime.strftime(item["date"], "%d-%b-%y")
        print(
            f"  {index:>4} | {item['id']:<6} | {item['title']:<26.25} | {item['price']:>7.2f}  | {formatted_date:<13} | {f'{item["bookings"]}/{MAX_BOOKINGS}':<9}"
        )
    print("\n" + "_"*35 + " End of list " + "_"*35)

In [460]:
def select_movie():
    print("\n=== Select Movie ===\n")
    list_movies()
    print("\n0. Cancel Selection")
    while True:
        choice = prompt_choice("\nEnter ID of movie to select => ")
        if choice == 0:
            return None
        selected_movie = next(
            ((i, m) for i, m in enumerate(movies) if m["id"] == choice), None
        )
        if selected_movie is None:
            print(f"Movie of ID:[{choice}] not found. Try again.")
            continue
        return selected_movie


In [461]:
def remove_movie():
    selected_movie = select_movie()
    if selected_movie is None:
        return
    confirmed = prompt_confirm(
        f'Are you sure you want to delete "{selected_movie[1]["title"]}" ?'
    )
    if selected_movie[1]["bookings"] > 0 and confirmed:
        confirmed = prompt_confirm(
            f"The movie you are about to delete has been booked {selected_movie[1]['bookings']} times.\nDo you still want to proceed ?"
        )
    if confirmed:
        movies.pop(selected_movie[0])
    else:
        print("Cancelled")

In [462]:
def view_all(show_booked=True):
    print("\n=== All Movies ===\n")
    list_movies(show_booked)

In [463]:
def sort_movies_list(key, desc=False):
    movies.sort(key=lambda m: m[key], reverse=desc)
    print("Movies sorted")
    view_all()

In [464]:
def payment_confirmation(movie_index):
    actual_amount = movies[movie_index]["price"]
    payment_amount = prompt_float("Enter Payment amount to confirm => ")
    if payment_amount != actual_amount:
        print("Payment Failed")
        return False
    else:
        print("Payment Successful")
        return True


In [465]:
def book_ticket():
    while True:
        selected = select_movie()
        if selected is None:
            return
        if selected[1]["bookings"] < MAX_BOOKINGS:
            break
        print("This movie has been fully booked. Try another movie.")
    payment_confirmed = payment_confirmation(selected[0])
    if not payment_confirmed:
        print("Booking Cancelled")
        return
    selected[1]["bookings"] += 1
    print(f"Movie ticket for [{selected[1]['title']}] has been booked.")
    return


In [466]:
main_menu_list = [
    {"entry_name": "Admin", "action": lambda: run_menu(admin_menu_list, "Admin Mode")},
    {"entry_name": "User", "action": lambda: run_menu(user_menu_list, "User Mode")},
]

admin_menu_list = [
    {"entry_name": "Add Movie", "action": add_movie},
    {"entry_name": "Remove Movie", "action": remove_movie},
    {"entry_name": "View All Movies", "action": view_all},
    {
        "entry_name": "Sort Movies",
        "action": lambda: run_menu(sort_menu_list, "Sort Movies"),
    },
]

user_menu_list = [
    {
        "entry_name": "View All Movies",
        # view_all function is called with False as an argument to only 
        # display movies which are available for booking
        "action": lambda: view_all(show_booked=False),
    },
    {"entry_name": "Book Movie Ticket", "action": book_ticket},
    {
        "entry_name": "Sort Movies",
        "action": lambda: run_menu(sort_menu_list, "Sort Movies"),
    },
]

sort_menu_list = [
    {
        "entry_name": "Sort by Title (A to Z)",
        "action": lambda: sort_movies_list("title"),
    },
    {
        "entry_name": "Sort by Title (Z to A)",
        "action": lambda: sort_movies_list("title", desc=True),
    },
    {"entry_name": "Sort by Earliest", "action": lambda: sort_movies_list("date")},
    {
        "entry_name": "Sort by Latest",
        "action": lambda: sort_movies_list("date", desc=True),
    },
    {
        "entry_name": "Sort by Price (Lowest)",
        "action": lambda: sort_movies_list("price"),
    },
    {
        "entry_name": "Sort by Price (Highest)",
        "action": lambda: sort_movies_list("price", desc=True),
    },
]

In [467]:
def main():
    run_menu(main_menu_list, "Main Menu")
    print("\nExiting Application")


if __name__ == "__main__":
    main()


 Main Menu
 1. Admin
 2. User
 0. Back
___________________________________
 Admin Mode
 1. Add Movie
 2. Remove Movie
 3. View All Movies
 4. Sort Movies
 0. Back
___________________________________

=== All Movies ===

    Sn | ID     | Title                      |   Price | Date          | Bookings 
     1 | 1000   | Inception                  |   10.00  | 01-May-26     | 0/5      
     2 | 1001   | The Dark Knight            |   12.50  | 02-May-26     | 0/5      
     3 | 1002   | Interstellar               |   11.00  | 03-May-26     | 5/5      
     4 | 1003   | Avengers: Endgame          |   13.00  | 04-May-26     | 5/5      
     5 | 1004   | Spider-Man: No Way Home    |   12.00  | 05-May-26     | 0/5      

___________________________________ End of list ___________________________________
 Admin Mode
 1. Add Movie
 2. Remove Movie
 3. View All Movies
 4. Sort Movies
 0. Back
 Main Menu
 1. Admin
 2. User
 0. Back
___________________________________
 User Mode
 1. View All Movie